# Bitcoin Long Accumulation Strategy: Composite On-Chain Signal Index

Below is an implementation of **composite on-chain signal index** to build a long-only, budget-constrained Bitcoin accumulation strategy. The goal is to check if this strategy outperforms uniform Dollar-Cost Averaging (DCA).

### Signal Dimensions

| Signal | Dimension | When to buy |
|---|---|---|
| `mvrv` | Market/realised-value ratio — valuation | Low mvrv = market undervaluation |
| `sopr_7d_ema`| Realised profit/loss of spending | sopr < 1 = investors selling at loss |
| `reserve_risk` | Holder conviction vs. price | Low reserve risk = high confidence, good buy zone |
| `greed_index` | Crowd sentiment | 0 = extreme fear, potential bottoms |
| `puell_multiple` | Miner economics | Low puell multiple = miner stress/capitulation |
| `lth_nupl` | Long-term holder unrealised P&L | Low lth nupl = capitualation/undervaluation |
| `sell_side_risk_ratio_7d_ema` | Short-term selling pressure | Low = market participants not eager to sell |
| `net_unrealized_pnl_rel_to_market_cap` | Aggregate unrealised P&L (NUPL) | Low = investors under water |

### Strategy Logic

Each of **8 on-chain signals** is rolling-z-scored (365-day window, fully causal), inverted into a *cheapness score* (the strategy wants a high score on cheap days. But cheap days produce negative z-scores. So simply negating flips the relationship), and blended via a **softmax-weighted sum**:

$$\text{cheapness}_t = \sum_{i=1}^{8} w_i \cdot \left(-z_t^{(i)}\right), \qquad w_i = \frac{e^{\ell_i}}{\sum_j e^{\ell_j}}$$

$$\text{buy\_score}_t = \max(0,\; \text{cheapness}_t)^{\gamma}$$

The logit vector $\boldsymbol{\ell} \in \mathbb{R}^{8}$ and exponent $\gamma > 0$ are jointly optimised by **Nelder-Mead** on the training period to maximise sats-per-dollar (SPD) relative to uniform DCA.

### Core Constraints
- **365-day rolling window** = budget resets to 1.0 every 365 days (non-overlapping).
- **Sum of weights = 1** per window (fixed total budget).
- **Causal execution** = weight at day $t$ uses only data up to day $t$.
- **Per-day bounds** = $w_t \in [10^{-5},\; 0.1]$.
- **Benchmark** = Uniform DCA: $w_t = 1/365$ for all $t$.

In [1]:
import sys
import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import softmax as scipy_softmax
from scipy.optimize import minimize
from pathlib import Path
from matplotlib import pyplot as plt

_root = Path.cwd()
while not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src import config, data_utils, plots, strategy_utils

TRAIN_PATH = config.TRAIN_PATH
#STACKSATS_DATA_PATH = config.STACKSATS_DATA_PATH
RAW_PATH = config.RAW_PATH

#data_utils.check_stacksats_data(STACKSATS_DATA_PATH, RAW_PATH)

# --- Configuration & Time Boundaries ---
TRAIN_START = "2018-01-01"
TRAIN_END   = "2023-12-31"
TEST_START  = "2024-01-01"
TEST_END    = "2025-12-31"
WINDOW_SIZE = 365
MIN_WEIGHT  = 1e-5
MAX_WEIGHT  = 0.1

# --- Signal columns (8 on-chain metrics to weight) ---
SIGNAL_COLS = [
    "mvrv",
    # "adjusted_sopr", 
    # "adjusted_sopr_7d_ema",
    # "realized_cap_growth_rate",
    # "market_cap_growth_rate",
    "sopr_7d_ema",
    "reserve_risk",
    "greed_index",
    "puell_multiple",
    "lth_nupl",
    "sell_side_risk_ratio_7d_ema",
    "net_unrealized_pnl_rel_to_market_cap",
]
N_SIGNALS = len(SIGNAL_COLS)
Z_COLS    = [f"z_{c}" for c in SIGNAL_COLS]

# Metrics needed from parquet (include market_cap + supply_btc to derive price)
LOAD_METRICS = (
    "market_cap",
    "supply_btc",
    *SIGNAL_COLS,
)

# --- Load training data ---
schema  = pl.read_parquet_schema(TRAIN_PATH)
columns = set(schema)

df = (
    pl.scan_parquet(TRAIN_PATH)
    .filter(pl.col("metric").is_in(LOAD_METRICS))
    .select("day_utc", "metric", "value")
    .collect()
    .pivot(values="value", index="day_utc", on="metric")
    .with_columns((pl.col("market_cap") / pl.col("supply_btc")).alias("price_usd"))
    .rename({"day_utc": "date"})
    .select(["date", "price_usd"] + SIGNAL_COLS)
    .filter(pl.col("price_usd").is_finite() & (pl.col("price_usd") > 0))
    .sort("date")
)

print(f"Train rows loaded: {len(df)}")
df.head()

Train rows loaded: 4886


date,price_usd,mvrv,sopr_7d_ema,reserve_risk,greed_index,puell_multiple,lth_nupl,sell_side_risk_ratio_7d_ema,net_unrealized_pnl_rel_to_market_cap
date,f64,f64,f64,f64,f64,f64,f64,f64,f64
2010-08-16,0.06,75.25938,0.644378,0.029851,0.0,364.99997,55.39583,21.301033,98.671234
2010-08-17,0.07,56.812218,0.860891,0.033654,0.01,184.63878,55.45579,22.381466,98.23979
2010-08-18,0.07,37.07454,1.048291,0.028037,0.01,125.12996,47.554413,20.86954,83.40232
2010-08-19,0.07,32.22305,1.152911,0.027273,0.01,94.20886,47.629772,17.275007,83.05423
2010-08-20,0.07,28.2621,1.161407,0.026549,0.01,98.38138,47.659138,13.974726,82.681435


## Step 1 - Data Preprocessing: Causal Rolling Z-Scores

Each on-chain signal is standardised using a **causal rolling 365-day z-score**:

$$z_t^{(i)} = \frac{x_t^{(i)} - \mu_{t,365}^{(i)}}{\sigma_{t,365}^{(i)} + \varepsilon}$$

where $\mu_{t,365}$ and $\sigma_{t,365}$ are the mean and standard deviation computed over the trailing 365-day window ending at day $t$ (`min_periods=2` to handle the warm-up period). A small constant $\varepsilon = 10^{-8}$ prevents division by zero.

- **Train period**: 2018-01-01 -> 2023-12-31  
- **Test period**: 2024-01-01 -> 2025-12-31  
- Z-scores are computed on the **combined** dataset so that the rolling window is continuous across the train/test boundary (no data leakage - each day's z-score only uses its own trailing window).

In [2]:
# Load, pivot, and derive price from long-format parquet
def load_and_pivot(path: str) -> pd.DataFrame:
    """Load a long-format parquet, pivot to wide, derive price_usd."""
    return (
        pl.scan_parquet(path)
        .filter(pl.col("metric").is_in(LOAD_METRICS))
        .select("day_utc", "metric", "value")
        .collect()
        .pivot(values="value", index="day_utc", on="metric")
        .with_columns((pl.col("market_cap") / pl.col("supply_btc")).alias("price_usd"))
        .rename({"day_utc": "date"})
        .select(["date", "price_usd"] + SIGNAL_COLS)
        .filter(pl.col("price_usd").is_finite() & (pl.col("price_usd") > 0))
        .sort("date")
        .to_pandas()
    )

# Causal rolling 365-day z-score 
def rolling_zscore(series: pd.Series, window: int = 365) -> pd.Series:
    """Compute rolling z-score with a causal window."""
    roll = series.rolling(window=window, min_periods=2)
    return (series - roll.mean()) / (roll.std() + 1e-8)

def trim_full_windows(df: pd.DataFrame, window_size: int = WINDOW_SIZE):
    n_full = len(df) // window_size
    n_eval = n_full * window_size
    remainder = len(df) - n_eval
    return df.iloc[:n_eval].copy(), n_full, remainder

combined_raw = load_and_pivot(RAW_PATH)
combined_raw["date"] = pd.to_datetime(combined_raw["date"])

print(f"Combined dataset: {len(combined_raw)} rows  "
      f"({combined_raw['date'].min().date()} → {combined_raw['date'].max().date()})")
print(f"Missing values per signal:\n{combined_raw[SIGNAL_COLS].isna().sum().to_string()}")

combined = combined_raw.copy()
for col in SIGNAL_COLS:
    combined[f"z_{col}"] = rolling_zscore(combined[col], window=WINDOW_SIZE)

# Forward-fill to handle sparse signals (e.g. reserve_risk gaps), Strict causality: no backward fill
combined[Z_COLS] = combined[Z_COLS].ffill()

# Train / Test splits
train_df = combined[
    (combined["date"] >= TRAIN_START) & (combined["date"] <= TRAIN_END)
].reset_index(drop=True)

test_df = combined[
    (combined["date"] >= TEST_START) & (combined["date"] <= TEST_END)
].reset_index(drop=True)

# Keep rows that are fully usable for scoring
train_df = train_df.dropna(subset=["price_usd"] + Z_COLS).sort_values("date").reset_index(drop=True)
test_df  = test_df.dropna(subset=["price_usd"] + Z_COLS).sort_values("date").reset_index(drop=True)

# Full-window evaluation sets only
train_eval_df, n_train_windows, train_remainder = trim_full_windows(train_df, WINDOW_SIZE)
test_eval_df, n_test_windows, test_remainder = trim_full_windows(test_df, WINDOW_SIZE)

print(f"Train rows total={len(train_df)}, full-window eval rows={len(train_eval_df)}, windows={n_train_windows}, remainder={train_remainder}")
print(f"Test  rows total={len(test_df)},  full-window eval rows={len(test_eval_df)},  windows={n_test_windows}, remainder={test_remainder}")

#print(f"\nTrain: {len(train_df)} rows | Test: {len(test_df)} rows")
print(f"NaN in z-scored train signals: {train_df[Z_COLS].isna().sum().sum()}")
train_eval_df[["date", "price_usd"] + Z_COLS].head(10)

Combined dataset: 5689 rows  (2010-08-16 → 2026-03-13)
Missing values per signal:
mvrv                                    0
sopr_7d_ema                             0
reserve_risk                            0
greed_index                             0
puell_multiple                          0
lth_nupl                                0
sell_side_risk_ratio_7d_ema             0
net_unrealized_pnl_rel_to_market_cap    0
Train rows total=2191, full-window eval rows=2190, windows=6, remainder=1
Test  rows total=731,  full-window eval rows=730,  windows=2, remainder=1
NaN in z-scored train signals: 0


,date,price_usd,z_mvrv,z_sopr_7d_ema,z_reserve_risk,z_greed_index,z_puell_multiple,z_lth_nupl,z_sell_side_risk_ratio_7d_ema,z_net_unrealized_pnl_rel_to_market_cap
0,2018-01-01,13466.31,-0.018563,-0.453264,1.519986,2.409743,0.874200,-0.240062,0.642310,0.183284
1,2018-01-02,14888.11,0.445743,-0.300535,1.808096,2.157409,1.402588,-0.217997,0.567724,0.559715
2,2018-01-03,15098.14,0.550742,-0.303331,1.857087,2.114028,1.518643,-0.015153,0.403214,0.750146
3,2018-01-04,15144.99,0.519210,-0.182288,1.825269,1.974431,1.649768,0.008468,0.414528,0.720643
4,2018-01-05,16960.01,1.081544,0.180665,2.193030,2.152453,1.304836,0.091369,0.603476,1.114781
5,2018-01-06,17115.00,1.070447,0.247602,2.178842,2.069565,1.868143,0.167870,0.586168,1.115438
6,2018-01-07,16174.22,0.623257,0.529704,1.904783,2.174020,1.665734,0.010288,0.772679,0.757300
7,2018-01-08,15000.00,0.203236,0.067870,1.605367,2.030056,1.415335,-0.011109,0.593532,0.411809
8,2018-01-09,14427.01,-0.013129,-0.044856,1.452929,2.231179,1.313353,0.019846,0.487306,0.229108
9,2018-01-10,14789.07,0.129649,-0.267004,1.518681,1.899709,1.105382,0.128199,0.384306,0.415337


## Step 2 - Composite On-Chain Signal Index

### Formula

**Cheapness score** (high = market is cheap / under-valued):

$$\text{cheapness}_t = \sum_{i=1}^{8} w_i \cdot \left(-z_t^{(i)}\right)$$

Signals that measure *market heat* (e.g. MVRV, SOPR, greed index) have **positive** z-scores when the market is expensive, so their negation flips them into cheapness contributions.

**Buy score** (non-negative, amplified by exponent):

$$\text{buy\_score}_t = \max\!\left(0,\; \text{cheapness}_t\right)^{\gamma}$$

- When $\text{cheapness}_t \leq 0$ (expensive market) the score is zero - no amplification above baseline.  
- $\gamma > 1$ concentrates capital on the *deepest* cheap periods; $\gamma < 1$ smooths contributions.



In [3]:
def compute_scores(df: pd.DataFrame, logits: np.ndarray, exponent: float) -> np.ndarray:
    """
    Compute the composite buy_score for each row in df.

    Parameters
    ----------
    df       : DataFrame with columns z_{signal} for every signal in SIGNAL_COLS.
    logits   : shape (N_SIGNALS,) — raw logit values; softmax gives the signal weights.
    exponent : float > 0 — sharpness of the buy score.

    Returns
    -------
    buy_score : np.ndarray shape (len(df),), values >= 0.
    """
    weights   = scipy_softmax(logits)                     # (8,) — sums to 1
    z_matrix  = df[Z_COLS].fillna(0.0).to_numpy()         # (N, 8)
    cheapness = z_matrix @ (-weights)                     # (N,) — high = cheap
    buy_score = np.maximum(0.0, cheapness) ** exponent
    return buy_score


# Sanity check with uniform logits (equal weights) and exponent=1 
_logits0   = np.zeros(N_SIGNALS)
_exponent0 = 1.0
_scores0   = compute_scores(train_df, _logits0, _exponent0)

print(f"Uniform-weight buy scores on train set:")
print(f"  Non-zero fraction : {(_scores0 > 0).mean():.2%}")
print(f"  Mean (non-zero)   : {_scores0[_scores0 > 0].mean():.4f}")
print(f"  Max               : {_scores0.max():.4f}")
print(f"  Min (non-zero)    : {_scores0[_scores0 > 0].min():.6f}")


Uniform-weight buy scores on train set:
  Non-zero fraction : 57.33%
  Mean (non-zero)   : 0.9506
  Max               : 2.2538
  Min (non-zero)    : 0.001674


## Step 3 - Online Causal Budget Allocation

### Constraints
| Constraint | Value |
|---|---|
| Sum of weights per window | $= 1$ |
| Min daily weight | $\geq 10^{-5}$ |
| Max daily weight | $\leq 0.1$ |
| Past weights | Immutable once set |
| Future information | Never used |

### Algorithm (per 365-day window)

```
remaining_budget ← 1.0
running_scores   ← []

for t = 0 … 364:
    score_t  ← buy_score[t]          # causal: only uses data ≤ t
    running_scores.append(score_t)
    remaining_days_after ← 364 − t

    if t == 364:                      # last day — consume all remaining budget
        w_t ← remaining_budget

    else:
        running_mean ← mean(running_scores)
        estimated_remaining_sum ← score_t + running_mean × remaining_days_after

        if estimated_remaining_sum > 0:
            w_t_raw ← (score_t / estimated_remaining_sum) × remaining_budget
        else:
            w_t_raw ← remaining_budget / (remaining_days_after + 1)   # uniform fallback

        max_allowed ← min(MAX_WEIGHT, remaining_budget − remaining_days_after × MIN_WEIGHT)
        w_t ← clamp(w_t_raw, MIN_WEIGHT, max(max_allowed, MIN_WEIGHT))

    remaining_budget −= w_t

# Post-process: iterative capping projection to enforce [MIN_WEIGHT, MAX_WEIGHT]
# and renormalise to sum = 1 exactly.
```

**Key insight** — the denominator `score_t + running_mean × remaining_days_after` estimates the expected total score of the remaining window using only past observations (running mean). This is the simplest unbiased estimator available at day $t$ without look-ahead.

The `max_allowed` guard ensures that after taking $w_t$, sufficient budget remains to give every future day at least `MIN_WEIGHT`.

In [4]:
def _project_weights(w: np.ndarray, min_w: float, max_w: float, eps: float = 1e-12) -> np.ndarray:
    """
    Project onto bounded simplex:
      sum(w)=1, min_w <= w_i <= max_w
    with early infeasibility checks and no final renormalization.
    """
    x = np.asarray(w, dtype=float).copy()
    n = x.size

    if n == 0:
        raise ValueError("Empty window.")
    if n * max_w < 1 - eps: 
        raise ValueError(f"Infeasible bounds: n*max_w={n*max_w:.6f} < 1.")
    if n * min_w > 1 + eps:
        raise ValueError(f"Infeasible bounds: n*min_w={n*min_w:.6f} > 1.")

    lo = np.full(n, min_w, dtype=float)
    hi = np.full(n, max_w, dtype=float)
    x = np.clip(x, lo, hi)

    fixed = np.zeros(n, dtype=bool)

    for _ in range(n + 10):
        free = ~fixed # bitwise NOT to get free indices        
        target = 1.0 - x[fixed].sum()

        if target < -eps:
            raise ValueError("Infeasible during projection: fixed weights exceed budget.")
        if free.sum() == 0:
            break

        base = x[free]
        base_sum = base.sum()
        if base_sum <= eps:
            base = np.full(free.sum(), 1.0 / free.sum())
        else:
            base = base / base_sum

        x[free] = base * target

        low = x < lo - eps
        high = x > hi + eps
        violated = low | high
        if not violated.any():
            break

        x[low] = lo[low]
        x[high] = hi[high]
        fixed[low | high] = True

    # Final boundedness and exact-sum checks (no renormalization)
    if (x < lo - 1e-9).any() or (x > hi + 1e-9).any():
        raise ValueError("Projection failed bounds check.")
    if abs(x.sum() - 1.0) > 1e-8:
        # small correction through free-slack only
        resid = 1.0 - x.sum()
        slack = (x > lo + 1e-9) & (x < hi - 1e-9)
        if slack.any():
            x[slack] += resid / slack.sum()
        if (x < lo - 1e-9).any() or (x > hi + 1e-9).any() or abs(x.sum() - 1.0) > 1e-8:
            raise ValueError(f"Projection failed exact-sum check: sum={x.sum():.12f}")

    return x

def allocate_weights(
    scores: np.ndarray,
    window_size: int = WINDOW_SIZE,
    min_w: float = MIN_WEIGHT,
    max_w: float = MAX_WEIGHT,
) -> np.ndarray:
    """
    Ignore remainder everywhere: input length must be exactly multiple of window_size.
    """
    scores = np.asarray(scores, dtype=float)
    if (scores < 0).any():
        raise ValueError("Scores must be non-negative.")

    N = len(scores)
    if N % window_size != 0:
        raise ValueError(
            f"Scores length {N} not divisible by window_size={window_size}. "
            f"Trim to full windows before calling."
        )

    n_windows = N // window_size
    weights = np.zeros(N, dtype=float)

    for win_idx in range(n_windows):
        start = win_idx * window_size
        end = start + window_size
        win_scores = scores[start:end]

        remaining_budget = 1.0
        win_weights = np.zeros(window_size, dtype=float)
        running_sum = 0.0

        for i, score_t in enumerate(win_scores):
            remaining_days_after = window_size - 1 - i
            running_sum += score_t
            running_mean = running_sum / (i + 1)

            if i == window_size - 1:
                w_t = remaining_budget
            else:
                est_remaining = score_t + running_mean * remaining_days_after
                if est_remaining > 1e-12:
                    w_t_raw = (score_t / est_remaining) * remaining_budget
                else:
                    w_t_raw = remaining_budget / (remaining_days_after + 1)

                max_allowed = min(max_w, remaining_budget - remaining_days_after * min_w)
                max_allowed = max(max_allowed, min_w)
                w_t = float(np.clip(w_t_raw, min_w, max_allowed))

            win_weights[i] = w_t
            remaining_budget = max(remaining_budget - w_t, 0.0)

        weights[start:end] = _project_weights(win_weights, min_w, max_w)

    return weights

def verify_constraints_exact_eval(
    weights: np.ndarray,
    n_windows: int,
    window_size: int = WINDOW_SIZE,
    min_w: float = MIN_WEIGHT,
    max_w: float = MAX_WEIGHT,
    label: str = "",
    tol: float = 1e-8,
) -> None:
    """
    Verify exactly and only the evaluated full-window set.
    """
    expected_len = n_windows * window_size
    if len(weights) != expected_len:
        raise ValueError(f"{label} expected len={expected_len}, got {len(weights)}")

    errors = []
    for w in range(n_windows):
        s = weights[w * window_size:(w + 1) * window_size]
        if abs(s.sum() - 1.0) > tol:
            errors.append(f"Window {w+1}: sum={s.sum():.12f}")
        if s.min() < min_w - tol:
            errors.append(f"Window {w+1}: min={s.min():.12f} < {min_w}")
        if s.max() > max_w + tol:
            errors.append(f"Window {w+1}: max={s.max():.12f} > {max_w}")

    tag = f"[{label}] " if label else ""
    if errors:
        raise AssertionError(tag + "Constraint violations:\n" + "\n".join(errors))
    print(f"{tag}All evaluated windows pass constraints.")

## Step 4 - Nelder-Mead Optimization

### Parameter Space

| Parameter | Representation | Count |
|---|---|---|
| Signal logits $\ell_1 \dots \ell_{12}$ | Unconstrained reals; signal weights $= \text{softmax}(\boldsymbol{\ell})$ | 8 |
| Log-exponent $\log \gamma$ | Unconstrained real; $\gamma = e^{\log\gamma}$ clipped to $[e^{-3}, e^{3}]$ | 1 |
| **Total** |- | **9** |

Starting point $\boldsymbol{\theta}_0 = \mathbf{0}$ corresponds to **equal signal weights** and **$\gamma = 1$**.

### Objective

$$\max_{\boldsymbol{\theta}} \;\frac{1}{K}\sum_{k=1}^{K} \frac{\text{SPD}_{\text{strategy}}^{(k)}}{\text{SPD}_{\text{DCA}}^{(k)}}$$

where $K$ is the number of complete 365-day windows in the training period, and

$$\text{SPD}^{(k)} = \sum_{t \in \text{window } k} \frac{w_t}{P_t}$$

is the **sats-per-dollar** accumulated in window $k$ (higher = better).
Implemented as `minimize(-objective, θ, method="Nelder-Mead")`.

In [5]:
_train_prices = train_df["price_usd"].to_numpy()
_n_windows_train = len(train_df) // WINDOW_SIZE
_dca_w = np.full(WINDOW_SIZE, 1.0 / WINDOW_SIZE)
print(f"window size: {WINDOW_SIZE}, length of train_df: {len(train_df)}, number of windows: {_n_windows_train}")

def _spd_ratio(weights: np.ndarray, prices: np.ndarray, n_windows: int) -> float:
    """Mean per-window SPD ratio: strategy / uniform-DCA."""
    ratios = np.empty(n_windows, dtype=float)
    for k in range(n_windows):
        sl = slice(k * WINDOW_SIZE, (k + 1) * WINDOW_SIZE)
        p  = prices[sl]
        inv_p = 1.0 / p
        spd_strat = np.dot(weights[sl], inv_p)
        spd_dca   = np.dot(_dca_w,      inv_p)
        ratios[k] = spd_strat / (spd_dca + 1e-15)
    return float(ratios.mean())


_eval_counter = [0]


def objective(theta: np.ndarray) -> float:
    """Nelder-Mead objective — returns negative mean SPD ratio (to minimise)."""
    logits   = theta[:N_SIGNALS]
    exponent = float(np.exp(np.clip(theta[N_SIGNALS], -3.0, 3.0)))

    scores  = compute_scores(train_eval_df, logits, exponent)
    weights = allocate_weights(scores, window_size=WINDOW_SIZE, min_w=MIN_WEIGHT, max_w=MAX_WEIGHT)

    ratio = _spd_ratio(weights, _train_prices, _n_windows_train)

    _eval_counter[0] += 1
    if _eval_counter[0] % 500 == 0:
        print(f"  eval {_eval_counter[0]:>5d} | SPD ratio {ratio:.6f} | γ={exponent:.4f}")

    return -ratio   # minimise negative = maximise ratio


# Run Nelder-Mead 
print("Starting Nelder-Mead optimisation (up to 5 000 function evaluations)…\n")
theta0 = np.zeros(N_SIGNALS + 1)   # uniform logits + γ=1

result = minimize(
    objective,
    theta0,
    method="Nelder-Mead",
    options={
        "maxfev"  : 5000,
        "xatol"   : 1e-4, 
        "fatol"   : 1e-4, 
        "adaptive": True,      # scale simplex to parameter dimensionality
    },
)

opt_logits   = result.x[:N_SIGNALS]
opt_exponent = float(np.exp(np.clip(result.x[N_SIGNALS], -3.0, 3.0)))
opt_sig_w    = scipy_softmax(opt_logits)

print(f"\nOptimisation complete")
print(f"  Status              : {result.message}")
print(f"  Function evals      : {result.nfev}")
print(f"  Converged           : {result.success}")
print(f"  Best SPD ratio      : {-result.fun:.6f}  (+{(-result.fun - 1)*100:.2f}% vs DCA)")
print(f"  Optimal logits      : {opt_logits}")
print(f"  Optimal exponent γ  : {opt_exponent:.4f}")
print(f"\nLearned signal weights (softmax):")
for col, w in zip(SIGNAL_COLS, opt_sig_w):
    print(f"  {col:<45s} {w:.4f}")

window size: 365, length of train_df: 2191, number of windows: 6
Starting Nelder-Mead optimisation (up to 5 000 function evaluations)…

  eval   500 | SPD ratio 1.269602 | γ=20.0855
  eval  1000 | SPD ratio 1.279593 | γ=20.0855
  eval  1500 | SPD ratio 1.281404 | γ=20.0855
  eval  2000 | SPD ratio 1.306947 | γ=20.0855
  eval  2500 | SPD ratio 1.308503 | γ=20.0855
  eval  3000 | SPD ratio 1.314005 | γ=20.0855
  eval  3500 | SPD ratio 1.314300 | γ=20.0855
  eval  4000 | SPD ratio 1.314303 | γ=20.0855
  eval  4500 | SPD ratio 1.314307 | γ=20.0855
  eval  5000 | SPD ratio 1.314612 | γ=20.0855

Optimisation complete
  Status              : Maximum number of function evaluations has been exceeded.
  Function evals      : 5000
  Converged           : False
  Best SPD ratio      : 1.314612  (+31.46% vs DCA)
  Optimal logits      : [ 2.77915856 -3.32998459 -3.08204987  0.97849262 -2.28960997  2.00355002
  0.23570355  2.35772062]
  Optimal exponent γ  : 20.0855

Learned signal weights (softmax):

In [6]:
# Learned signal weights bar chart
baseline_w = 1.0 / N_SIGNALS   # equal-weight reference

fig_w = go.Figure()
fig_w.add_trace(go.Bar(
    y=SIGNAL_COLS,
    x=opt_sig_w,
    orientation="h",
    text=[f"{w:.3f}" for w in opt_sig_w],
    textposition="outside",
    marker_color=[
        "steelblue" if w >= baseline_w else "lightcoral"
        for w in opt_sig_w
    ],
    name="Optimised weight",
))
fig_w.add_vline(
    x=baseline_w,
    line_dash="dash",
    line_color="black",
)
fig_w.add_annotation(
    x=baseline_w,
    y=1,              # > 1 pushes above plot area ("outside")
    xref="x",
    yref="paper",
    text=f"<b>Equal weight ({baseline_w:.3f})</b>",
    showarrow=False,     
    xanchor="left",
    yanchor="bottom",
)
fig_w.update_xaxes(
    tickfont=dict(size=16, color="black")
)
fig_w.update_yaxes(
    tickfont=dict(family="Arial Black", size=16, color="black")
)
fig_w.update_layout(
    title=f"Optimised Signal Weights  (γ = {opt_exponent:.3f})",
    yaxis_title="On-chain signal",
    xaxis_title="Weight (softmax)",
    margin=dict(l=220, r=40, t=70, b=40),
    showlegend=False,
)
fig_w.update_yaxes(autorange="reversed")
fig_w.show()

## Step 5 - Backtest: Train Period (2018 – 2023)

We evaluate the strategy on the training set using the optimised parameters.  
**SPD improvement** per window = `(strategy_SPD / DCA_SPD) − 1`.  
A positive value means the strategy accumulated **more BTC per dollar spent** than uniform DCA.

In [7]:
def backtest(df_eval: pd.DataFrame, logits: np.ndarray, exponent: float, label: str = ""):
    """
    df_eval must already be trimmed to full windows only.
    """
    n_windows = len(df_eval) // WINDOW_SIZE
    if n_windows == 0:
        raise ValueError(f"{label} has no full 365-day windows to evaluate.")

    if len(df_eval) != n_windows * WINDOW_SIZE:
        raise ValueError(f"{label} df_eval is not full-window aligned.")

    scores = compute_scores(df_eval, logits, exponent)
    weights = allocate_weights(scores, window_size=WINDOW_SIZE, min_w=MIN_WEIGHT, max_w=MAX_WEIGHT)
    verify_constraints_exact_eval(weights, n_windows=n_windows, label=label)

    prices = df_eval["price_usd"].to_numpy()
    dates = df_eval["date"].to_numpy()
    dca_w_win = np.full(WINDOW_SIZE, 1.0 / WINDOW_SIZE)

    rows = []
    for k in range(n_windows):
        sl = slice(k * WINDOW_SIZE, (k + 1) * WINDOW_SIZE)
        inv_p = 1.0 / prices[sl]
        rows.append(
            {
                "window": k + 1,
                "start_date": pd.Timestamp(dates[sl.start]).date(),
                "end_date": pd.Timestamp(dates[sl.stop - 1]).date(),
                "strategy_btc": float(np.dot(weights[sl], inv_p)),
                "dca_btc": float(np.dot(dca_w_win, inv_p)),
            }
        )

    results_df = pd.DataFrame(rows)
    results_df["improvement_pct"] = (results_df["strategy_btc"] / results_df["dca_btc"] - 1.0) * 100.0

    results_df["win"] = (results_df["strategy_btc"] > results_df["dca_btc"]).astype(int)
    results_df["excess_btc"] = results_df["strategy_btc"] - results_df["dca_btc"]

    win_rate = results_df["win"].mean()
    loss_rate = 1.0 - win_rate
    avg_win_excess = results_df.loc[results_df["win"] == 1, "excess_btc"].mean()
    avg_loss_excess = results_df.loc[results_df["win"] == 0, "excess_btc"].mean()
    median_improvement_pct = results_df["improvement_pct"].median()

    metrics = {
    "n_windows": int(len(results_df)),
    "win_rate": float(win_rate),
    "loss_rate": float(loss_rate),
    "wins": int(results_df["win"].sum()),
    "losses": int((1 - results_df["win"]).sum()),
    "avg_win_excess_btc": float(avg_win_excess) if pd.notna(avg_win_excess) else 0.0,
    "avg_loss_excess_btc": float(avg_loss_excess) if pd.notna(avg_loss_excess) else 0.0,
    "median_improvement_pct": float(median_improvement_pct),
    }

    print(f" Win rate (window-level): {metrics['win_rate']:.2%} ({metrics['wins']}/{metrics['n_windows']})")
    
    return results_df, scores, weights, metrics

train_results, train_scores, train_weights, train_metrics = backtest(train_eval_df, opt_logits, opt_exponent, label="TRAIN")
#test_results, test_scores, test_weights, test_metrics = backtest(test_eval_df, opt_logits, opt_exponent, label="TEST")

[TRAIN] All evaluated windows pass constraints.
 Win rate (window-level): 83.33% (5/6)


In [8]:
# Test set evaluation (frozen optimal parameters)
# Z-scores for the test set are already computed from the combined dataset,
# so they continue the rolling window seamlessly from the training period.
test_results, test_scores, test_weights, test_metrics = backtest(
    test_eval_df, opt_logits, opt_exponent, label="TEST (2024-2025)"
)

[TEST (2024-2025)] All evaluated windows pass constraints.
 Win rate (window-level): 100.00% (2/2)


In [9]:
def add_exp_decay_percentile(results_df: pd.DataFrame, score_col: str = "improvement_pct", decay: float = 0.9, eps: float = 1e-12):
    df = results_df.copy().reset_index(drop=True)
    r = df[score_col].to_numpy(dtype=float)

    # Expanding percentile: each window scored within history up to that window
    p = np.zeros_like(r, dtype=float)
    for t in range(len(r)):
        hist = r[: t + 1]
        lo = float(np.min(hist))
        hi = float(np.max(hist))
        p[t] = 100.0 * (r[t] - lo) / (hi - lo + eps)

    # Recency-biased weighted average
    w = decay ** np.arange(len(r) - 1, -1, -1, dtype=float)
    exp_decay_score = float(np.dot(w, p) / np.sum(w)) if len(r) > 0 else np.nan

    df["window_percentile"] = p
    return df, exp_decay_score


train_results, train_decay_score = add_exp_decay_percentile(train_results, score_col="improvement_pct", decay=0.9)
test_results, test_decay_score = add_exp_decay_percentile(test_results, score_col="improvement_pct", decay=0.9)


In [10]:
# Full-period visualisation (4 panels)
all_scores  = np.concatenate([train_scores, test_scores])
all_weights = np.concatenate([train_weights, test_weights])
all_dates   = pd.to_datetime(
    np.concatenate([train_eval_df["date"].to_numpy(), test_eval_df["date"].to_numpy()])
)
all_prices  = np.concatenate([
    train_eval_df["price_usd"].to_numpy(),
    test_eval_df["price_usd"].to_numpy(),
])

dca_weight  = 1.0 / WINDOW_SIZE
cum_strat   = np.cumsum(all_weights / all_prices)
cum_dca     = np.cumsum(np.full(len(all_weights), dca_weight) / all_prices)

# Per-window improvement bars (train + test)
all_win_df = pd.concat([
    train_results.assign(period="Train"),
    test_results.assign(period="Test"),
], ignore_index=True)
all_win_df, overall_decay_score = add_exp_decay_percentile(all_win_df, score_col="improvement_pct", decay=0.9)


overall_wins = train_metrics["wins"] + test_metrics["wins"]
overall_n = train_metrics["n_windows"] + test_metrics["n_windows"]
overall_win_rate = overall_wins / overall_n if overall_n > 0 else np.nan
print(f"Overall win rate: {overall_win_rate:.2%} ({overall_wins}/{overall_n})")
print(f"Train exp_decay_percentile (base=0.9): {train_decay_score:.2f}")
print(f"Test exp_decay_percentile (base=0.9): {test_decay_score:.2f}")
print(f"Overall exp_decay_percentile(base=0.9): {overall_decay_score:.2f}")

# bar_labels = [
#     f"W{r.window} {str(r.start_date)[:7]}"
#     for r in all_win_df.itertuples()
# ]
# bar_colors = [
#     "steelblue" if p == "Train" else "seagreen"
#     for p in all_win_df["period"]
# ]

bar_labels = [
    f"W{r.window} {str(r.start_date)[:7]}"
    for r in all_win_df.itertuples()
]
bar_colors = [
    "steelblue" if p == "Train" else "seagreen"
    for p in all_win_df["period"]
]
bar_text = [
    f"{imp:.1f}% | Pctl {pct:.1f}"
    for imp, pct in zip(all_win_df["improvement_pct"], all_win_df["window_percentile"])
]

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=[
        "Composite Buy Score (cheapness index)",
        "Daily Allocation Weight — Strategy vs. Uniform DCA",
        "Cumulative BTC Accumulated: Strategy vs. Uniform DCA",
        "Per-Window SPD Improvement over Uniform DCA (%)",
    ],
    vertical_spacing=0.08,
    row_heights=[0.18, 0.18, 0.36, 0.28],
    shared_xaxes=False,
)

# Panel 1 - buy score
fig.add_trace(
    go.Scatter(x=all_dates, y=all_scores,
               name="Buy score", line=dict(color="goldenrod", width=1)),
    row=1, col=1,
)

# Panel 2 - weights
fig.add_trace(
    go.Scatter(x=all_dates, y=all_weights,
               name="Strategy weight", line=dict(color="steelblue", width=0.8),
               fill="tozeroy", fillcolor="rgba(70,130,180,0.15)"),
    row=2, col=1,
)
fig.add_trace(
    go.Scatter(
        x=[all_dates.min(), all_dates.max()],
        y=[dca_weight, dca_weight],
        name=f"Uniform DCA (1/365 = {dca_weight:.5f})",
        line=dict(color="red", dash="dot", width=1.5),
    ),
    row=2, col=1,
)

# Panel 3 - cumulative BTC
fig.add_trace(
    go.Scatter(x=all_dates, y=cum_strat,
               name="Strategy", line=dict(color="steelblue", width=2)),
    row=3, col=1,
)
fig.add_trace(
    go.Scatter(x=all_dates, y=cum_dca,
               name="Uniform DCA", line=dict(color="red", dash="dot", width=1.5)),
    row=3, col=1,
)

# Train / test boundary shading on panels 1–3
for row_i in [1, 2, 3]:
    fig.add_vrect(
        x0=TEST_START, x1=str(all_dates.max().date()),
        fillcolor="lightgreen", opacity=0.15, line_width=0,
        annotation_text="Test" ,
        annotation_position="top left",
        row=row_i, col=1,
    )

# Panel 4 - per-window SPD improvement bars
fig.add_trace(
    go.Bar(
        x=bar_labels,
        y=all_win_df["improvement_pct"],
        marker_color=bar_colors,
        # text=[f"{v:.1f}%" for v in all_win_df["improvement_pct"]],
        # textposition="outside",
        # name="SPD improvement %",
        text=bar_text,
        textposition="outside",
        customdata=np.c_[all_win_df["window_percentile"].to_numpy(), all_win_df["period"].to_numpy()],
        hovertemplate=(
            "<b>%{x}</b><br>"
            "SPD improvement: %{y:.2f}%<br>"
            "Expanding percentile: %{customdata[0]:.2f}<br>"
            "Period: %{customdata[1]}<extra></extra>"
        ),
        name="SPD improvement %",
    ),
    row=4, col=1,
)
fig.add_hline(y=0, line_color="black", line_width=1, row=4, col=1)

# Layout
fig.update_layout(
    title=dict(
        text=(
            f"Composite On-Chain Signal Strategy — Full Backtest<br>"
            f"<sup>γ={opt_exponent:.3f}  |  Train improvement: "
            f"{(train_results['strategy_btc'].sum()/train_results['dca_btc'].sum()-1)*100:.2f}%  |  "
            f"Test improvement: "
            f"{(test_results['strategy_btc'].sum()/test_results['dca_btc'].sum()-1)*100:.2f}%</sup>"
        ),
        x=0.5,
    ),
    height=1700,
    legend=dict(orientation="h", y=-0.02),
    margin=dict(l=60, r=30, t=100, b=60),
)
fig.update_yaxes(title_text="Buy score",   row=1, col=1)
fig.update_yaxes(title_text="Weight",      row=2, col=1)
fig.update_yaxes(title_text="BTC / $1 invested", row=3, col=1)
fig.update_yaxes(title_text="Improvement %",     row=4, col=1)
fig.update_xaxes(title_text="Date",              row=4, col=1)

fig.show()

Overall win rate: 87.50% (7/8)
Train exp_decay_percentile (base=0.9): 34.31
Test exp_decay_percentile (base=0.9): 52.63
Overall exp_decay_percentile(base=0.9): 31.44


In [11]:
# stacksats imports — adjust path if your environment differs
from stacksats.strategy_types import BaseStrategy, DayState

class CompositeSignalIndex(BaseStrategy):
    """
    Composite On-Chain Signal Index — long-only BTC accumulation strategy.

    Each of N on-chain signals is rolling-z-scored (externally, causally),
    blended via softmax-weighted cheapness score, then allocated through a
    causal online budget algorithm within each 365-day window.

    Signal weights (logits) and the sharpness exponent γ are jointly
    optimised by Nelder-Mead on the training period to maximise mean
    sats-per-dollar relative to uniform DCA.

    Parameters
    ----------
    full_zscored_df :
        Combined (train + test) pandas DataFrame with columns
        ``date``, ``price_usd``, and ``z_{signal}`` for every signal.
        Z-scores must already be computed causally (rolling 365-day).
    signal_cols :
        Ordered list of raw signal names (used to derive z-column names).
    min_weight, max_weight :
        Per-day allocation bounds (default 1e-5 / 0.1).
    window_size :
        Budget reset cadence in days (default 365).
    nelder_mead_maxfev :
        Max function evaluations for the optimiser.
    """

    strategy_id  = "composite-signal-index"
    version      = "1.0.0"
    description  = (
        "Softmax-blended on-chain cheapness index with "
        "Nelder-Mead optimised signal weights and sharpness exponent."
    )

    # Construction

    def __init__(
        self,
        full_zscored_df: pd.DataFrame,
        signal_cols: list[str] = SIGNAL_COLS,
        min_weight: float = MIN_WEIGHT,
        max_weight: float = MAX_WEIGHT,
        window_size: int  = WINDOW_SIZE,
        nelder_mead_maxfev: int = 5_000,
    ) -> None:
        self.signal_cols        = list(signal_cols)
        self.z_cols             = [f"z_{c}" for c in self.signal_cols]
        self.n_signals          = len(self.signal_cols)
        self.min_weight         = min_weight
        self.max_weight         = max_weight
        self.window_size        = window_size
        self.nelder_mead_maxfev = nelder_mead_maxfev

        # Store the full z-scored dataset (train + test)
        df = full_zscored_df.copy()
        df["date"] = pd.to_datetime(df["date"])
        self._full_df = df.sort_values("date").reset_index(drop=True)

        # Date → z-vector lookup for propose_weight (O(1) per day)
        self._zscore_lookup: dict[pd.Timestamp, np.ndarray] = {
            row["date"]: row[self.z_cols].to_numpy(dtype=float)
            for _, row in self._full_df.iterrows()
        }

        # Fitted parameters (populated by fit())
        self.opt_logits:   np.ndarray | None = None
        self.opt_exponent: float | None      = None
        self.fit_result_                     = None

        # Running state for propose_weight (reset each window)
        self._running_sum: float = 0.0

    # Fitting

    def fit(self, train_df: pd.DataFrame, verbose: bool = True) -> "CompositeSignalIndex":
        """
        Run Nelder-Mead optimisation on train_df to find the best signal
        logits and exponent γ (maximises mean SPD ratio vs uniform DCA).

        train_df must be a subset of full_zscored_df and include
        ``date``, ``price_usd``, and all ``z_{signal}`` columns.
        Rows are trimmed to an integer multiple of window_size internally.

        Returns self (chainable).
        """
        df = (
            train_df.copy()
            .assign(date=lambda d: pd.to_datetime(d["date"]))
            .dropna(subset=["price_usd"] + self.z_cols)
            .sort_values("date")
            .reset_index(drop=True)
        )

        n_full = len(df) // self.window_size
        if n_full == 0:
            raise ValueError(
                f"train_df has only {len(df)} rows — "
                f"need ≥{self.window_size} for one full window."
            )

        train_eval   = df.iloc[: n_full * self.window_size].copy()
        train_prices = train_eval["price_usd"].to_numpy()
        dca_w        = np.full(self.window_size, 1.0 / self.window_size)
        _counter     = [0]

        def _objective(theta: np.ndarray) -> float:
            logits   = theta[: self.n_signals]
            exponent = float(np.exp(np.clip(theta[self.n_signals], -3.0, 3.0)))
            scores   = self._compute_scores(train_eval, logits, exponent)
            weights  = self._allocate_weights_batch(scores)

            ratios = np.empty(n_full, dtype=float)
            for k in range(n_full):
                sl      = slice(k * self.window_size, (k + 1) * self.window_size)
                inv_p   = 1.0 / train_prices[sl]
                ratios[k] = np.dot(weights[sl], inv_p) / (np.dot(dca_w, inv_p) + 1e-15)

            _counter[0] += 1
            if verbose and _counter[0] % 500 == 0:
                γ = float(np.exp(np.clip(theta[self.n_signals], -3.0, 3.0)))
                print(f"  eval {_counter[0]:>5d} | SPD ratio {ratios.mean():.6f} | γ={γ:.4f}")

            return -float(ratios.mean())   # minimise → maximise SPD ratio

        if verbose:
            print("Starting Nelder-Mead optimisation …\n")

        result = minimize(
            _objective,
            x0     = np.zeros(self.n_signals + 1),   # uniform logits + γ=1
            method = "Nelder-Mead",
            options = {
                "maxfev"  : self.nelder_mead_maxfev,
                "xatol"   : 1e-4,
                "fatol"   : 1e-4,
                "adaptive": True,
            },
        )

        self.opt_logits   = result.x[: self.n_signals]
        self.opt_exponent = float(np.exp(np.clip(result.x[self.n_signals], -3.0, 3.0)))
        self.fit_result_  = result

        if verbose:
            print(f"\nOptimisation complete")
            print(f"  Status         : {result.message}")
            print(f"  Function evals : {result.nfev}")
            print(f"  Best SPD ratio : {-result.fun:.6f}  (+{(-result.fun - 1)*100:.2f}% vs DCA)")
            print(f"  γ              : {self.opt_exponent:.4f}")

        return self

    # BaseStrategy interface

    def params(self) -> dict[str, object]:
        return {
            "signal_cols"       : self.signal_cols,
            "min_weight"        : self.min_weight,
            "max_weight"        : self.max_weight,
            "window_size"       : self.window_size,
            "opt_logits"        : self.opt_logits.tolist() if self.opt_logits is not None else None,
            "opt_exponent"      : self.opt_exponent,
            "nelder_mead_maxfev": self.nelder_mead_maxfev,
        }

    def required_feature_columns(self) -> tuple[str, ...]:
        # Z-scores come from our own precomputed lookup;
        # we only need the framework to provide the date.
        return ()

    def propose_weight(self, state: DayState) -> float:
        """
        Causal online budget allocation for one day within a 365-day window.

        Called by the stacksats runner day-by-day.  Uses the optimised
        signal logits and exponent to compute today's buy_score, then
        allocates budget proportional to an estimated remaining score sum
        (running mean of observed scores — no look-ahead).

        Raises RuntimeError if fit() has not been called.
        """
        if self.opt_logits is None or self.opt_exponent is None:
            raise RuntimeError("Call fit(train_df) before running the strategy.")

        day_idx    = state.day_index    # 0-based within the current window
        total_days = state.total_days   # == window_size (365)
        remaining  = float(state.remaining_budget)

        # Reset running state at the start of each new window
        if day_idx == 0:
            self._running_sum = 0.0

        # Look up today's z-scores
        current_date = pd.Timestamp(state.current_date)
        z_vec = self._zscore_lookup.get(current_date)
        if z_vec is None:
            # Date not found — uniform fallback
            remaining_days = total_days - day_idx
            return float(np.clip(
                remaining / remaining_days,
                self.min_weight, self.max_weight,
            ))

        # Compute today's buy score
        sig_weights = scipy_softmax(self.opt_logits)             # (n_signals,)
        cheapness   = float(np.dot(-z_vec, sig_weights))
        buy_score   = float(max(0.0, cheapness) ** self.opt_exponent)

        self._running_sum += buy_score
        running_mean        = self._running_sum / (day_idx + 1)
        remaining_days_after = total_days - 1 - day_idx

        # Causal budget allocation
        if day_idx == total_days - 1:          # last day: spend remainder
            return float(remaining)

        est_remaining = buy_score + running_mean * remaining_days_after
        if est_remaining > 1e-12:
            w_raw = (buy_score / est_remaining) * remaining
        else:
            w_raw = remaining / (remaining_days_after + 1)

        max_allowed = min(
            self.max_weight,
            remaining - remaining_days_after * self.min_weight,
        )
        max_allowed = max(max_allowed, self.min_weight)

        return float(np.clip(w_raw, self.min_weight, max_allowed))

    # Plotting helper 

    def build_plot_df(
        self,
        eval_df: pd.DataFrame,
        weight_col: str = "dynamic_weight",
    ) -> pd.DataFrame:
        """
        Run the batch backtest on eval_df using the fitted parameters and
        return a pandas DataFrame compatible with
        ``plots.plot_strategy_full_period`` and ``plots.plot_strategy_by_year``.

        eval_df must already be trimmed to full windows (len % window_size == 0)
        and contain columns: date, price_usd, z_{signal}.

        The returned DataFrame has columns:
            date, price_usd,
            {weight_col}, baseline_weight,
            sats_per_dollar_dynamic, sats_per_dollar_baseline,
            sats_accum_dynamic, sats_accum_baseline,
            btc_accum_dynamic, btc_accum_baseline

        Pass a custom weight_col if you want to rename the strategy weight
        column (useful when comparing multiple strategies side-by-side).
        Use the matching StrategyColumns(weight=weight_col) when plotting.
        """
        if self.opt_logits is None or self.opt_exponent is None:
            raise RuntimeError("Call fit(train_df) before build_plot_df().")

        df = eval_df.copy()
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").reset_index(drop=True)

        scores  = self._compute_scores(df, self.opt_logits, self.opt_exponent)
        weights = self._allocate_weights_batch(scores)

        prices       = df["price_usd"].to_numpy()
        dca_w        = np.full(len(df), 1.0 / self.window_size)

        btc_dyn   = np.cumsum(weights   / prices)
        btc_base  = np.cumsum(dca_w     / prices)
        sats_dyn  = btc_dyn  * 1e8
        sats_base = btc_base * 1e8

        # Running sats-per-dollar: cumulative sats ÷ cumulative weight spent
        cum_w_dyn  = np.cumsum(weights)
        cum_w_base = np.cumsum(dca_w)
        spd_dyn    = np.where(cum_w_dyn  > 0, sats_dyn  / cum_w_dyn,  0.0)
        spd_base   = np.where(cum_w_base > 0, sats_base / cum_w_base, 0.0)

        plot_df = df[["date", "price_usd"]].copy()
        plot_df[weight_col]                     = weights
        plot_df["baseline_weight"]              = dca_w
        plot_df["sats_per_dollar_dynamic"]      = spd_dyn
        plot_df["sats_per_dollar_baseline"]     = spd_base
        plot_df["sats_accum_dynamic"]           = sats_dyn
        plot_df["sats_accum_baseline"]          = sats_base
        plot_df["btc_accum_dynamic"]            = btc_dyn
        plot_df["btc_accum_baseline"]           = btc_base

        return plot_df

    # Private helpers

    def _compute_scores(
        self,
        df: pd.DataFrame,
        logits: np.ndarray,
        exponent: float,
    ) -> np.ndarray:
        """Vectorised buy-score computation (mirrors notebook compute_scores)."""
        weights   = scipy_softmax(logits)
        z_matrix  = df[self.z_cols].fillna(0.0).to_numpy()
        cheapness = z_matrix @ (-weights)
        return np.maximum(0.0, cheapness) ** exponent

    def _allocate_weights_batch(self, scores: np.ndarray) -> np.ndarray:
        """
        Batch causal budget allocation across all full windows.
        Mirrors the notebook's allocate_weights() — used during fit() and
        build_plot_df() only (not during live propose_weight inference).
        """
        scores   = np.asarray(scores, dtype=float)
        N        = len(scores)
        n_wins   = N // self.window_size
        out      = np.zeros(N, dtype=float)

        for w in range(n_wins):
            s      = w * self.window_size
            e      = s + self.window_size
            win_sc = scores[s:e]

            remaining_budget = 1.0
            win_w            = np.zeros(self.window_size, dtype=float)
            running_sum      = 0.0

            for i, score_t in enumerate(win_sc):
                remaining_days_after = self.window_size - 1 - i
                running_sum += score_t
                running_mean = running_sum / (i + 1)

                if i == self.window_size - 1:
                    w_t = remaining_budget
                else:
                    est = score_t + running_mean * remaining_days_after
                    w_raw = (
                        (score_t / est) * remaining_budget
                        if est > 1e-12
                        else remaining_budget / (remaining_days_after + 1)
                    )
                    max_allowed = max(
                        min(self.max_weight, remaining_budget - remaining_days_after * self.min_weight),
                        self.min_weight,
                    )
                    w_t = float(np.clip(w_raw, self.min_weight, max_allowed))

                win_w[i] = w_t
                remaining_budget = max(remaining_budget - w_t, 0.0)

            out[s:e] = _project_weights(win_w, self.min_weight, self.max_weight)

        return out


In [14]:
from src.plots import StrategyColumns, plot_strategy_full_period, plot_strategy_by_year

# combined is the full z-scored df already built in the notebook
strategy = CompositeSignalIndex(full_zscored_df=combined)
strategy.fit(train_df)   # runs Nelder-Mead, stores opt_logits / opt_exponent

# Build the combined eval df (train + test full windows only)
full_eval_df = pd.concat([train_eval_df, test_eval_df], ignore_index=True)

# Build plot-compatible DataFrame
plot_df = strategy.build_plot_df(full_eval_df)

cols = StrategyColumns(
    weight    = "dynamic_weight",
    spd       = "sats_per_dollar_dynamic",
    sats_accum = "sats_accum_dynamic",
)

full_plot = plot_strategy_full_period(
    plot_df        = plot_df,
    cols           = cols,
    strategy_name  = "CSI",
    date_range     = (str(full_eval_df["date"].min().date()),
                      str(full_eval_df["date"].max().date())),
)

year_plot = plot_strategy_by_year(
    plot_df       = plot_df,
    cols          = cols,
    strategy_name = "CSI",
)

plt.show()

Starting Nelder-Mead optimisation …

  eval   500 | SPD ratio 1.269602 | γ=20.0855
  eval  1000 | SPD ratio 1.279593 | γ=20.0855
  eval  1500 | SPD ratio 1.281404 | γ=20.0855
  eval  2000 | SPD ratio 1.306947 | γ=20.0855
  eval  2500 | SPD ratio 1.308503 | γ=20.0855
  eval  3000 | SPD ratio 1.314005 | γ=20.0855
  eval  3500 | SPD ratio 1.314300 | γ=20.0855
  eval  4000 | SPD ratio 1.314303 | γ=20.0855
  eval  4500 | SPD ratio 1.314307 | γ=20.0855
  eval  5000 | SPD ratio 1.314612 | γ=20.0855

Optimisation complete
  Status         : Maximum number of function evaluations has been exceeded.
  Function evals : 5000
  Best SPD ratio : 1.314612  (+31.46% vs DCA)
  γ              : 20.0855


In [13]:
import math
from dataclasses import dataclass
from typing import Literal

# Data container 

@dataclass
class PlotData:
    """
    Holds the combined train+test DataFrame and per-period sats/$ summaries.
    Built once, reused by both plot functions.
    """
    df:              pd.DataFrame   # combined, with year/period/weight columns
    train_eval_df:    pd.DataFrame   # train rows only (for window slicing)
    train_weights:    np.ndarray     # train allocation weights
    test_eval_df:     pd.DataFrame   # test rows only (for window slicing)
    test_weights:     np.ndarray     # test allocation weights
    all_win_df:       pd.DataFrame   # combined backtest results (from backtest())
    train_strat_sats: float
    train_dca_sats:   float
    train_excess:     float
    train_excess_pct: float
    test_strat_sats:  float
    test_dca_sats:    float
    test_excess:      float
    test_excess_pct:  float
    train_decay_score: float
    test_decay_score: float
    overall_decay_score: float


# Shared helpers

def _sats_summary(
    prices: np.ndarray,
    strategy_weights: np.ndarray,
    dca_weights: np.ndarray,
    sats_per_btc: float = 100_000_000.0,
) -> tuple[float, float, float, float]:
    """
    Returns (strat_sats, dca_sats, excess, excess_pct) for a price/weight slice.
    """
    strat_sats = sats_per_btc * np.sum(strategy_weights / prices)
    dca_sats   = sats_per_btc * np.sum(dca_weights      / prices)
    excess     = strat_sats - dca_sats
    excess_pct = (excess / dca_sats * 100.0) if dca_sats > 0 else float("nan")
    return strat_sats, dca_sats, excess, excess_pct

def _get_window_slice(df_eval: pd.DataFrame, weights: np.ndarray, window_1based: int, window_size: int):
    start = (window_1based - 1) * window_size
    end = start + window_size
    df_slice = df_eval.iloc[start:end].reset_index(drop=True)
    w_slice = weights[start:end]
    if len(df_slice) != window_size or len(w_slice) != window_size:
        raise ValueError(
            f"Window {window_1based} is out of bounds for provided eval data/weights."
        )
    return df_slice, w_slice

def _arrow_text_html(excess_sats: float, excess_pct: float) -> str:
    """Coloured HTML arrow annotation for use in plot titles/subtitles."""
    if excess_sats >= 0:
        return (
            f"<span style='color:green;'>"
            f"<b>▲ +{excess_sats:,.2f} sats/$ ({excess_pct:+.2f}%)</b>"
            f"</span>"
        )
    return (
        f"<span style='color:red;'>"
        f"<b>▼ {excess_sats:,.2f} sats/$ ({excess_pct:+.2f}%)</b>"
        f"</span>"
    )


def _arrow_text_plain(excess_sats: float, excess_pct: float) -> str:
    """Plain-text arrow annotation for subplot titles (no HTML support)."""
    if excess_sats >= 0:
        return f"▲ +{excess_sats:,.1f} sats/$ ({excess_pct:+.2f}%)"
    return f"▼ {excess_sats:,.1f} sats/$ ({excess_pct:+.2f}%)"


def _add_weight_traces(
    fig,
    xdata:            pd.Series,
    price:            np.ndarray,
    strategy_weight:  np.ndarray,
    dca_weight:       np.ndarray,
    window_size:      int,
    row:              int,
    col:              int,
    show_legend:      bool = True,
) -> None:
    """
    Adds three traces to a subplot cell:
      1. BTC Price        (left y-axis, log scale)
      2. Strategy Weight  (right y-axis)
      3. Uniform DCA      (right y-axis, dashed red)
    """
    fig.add_trace(
        go.Scatter(
            x=xdata, y=price,
            name="BTC Price",
            mode="lines",
            line=dict(color="black", width=1.5),
            showlegend=show_legend,
            legendgroup="price",
            hovertemplate="<b>%{x|%Y-%m-%d}</b><br>Price: $%{y:,.0f}<extra></extra>",
        ),
        row=row, col=col, secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=xdata, y=strategy_weight,
            name="Strategy Weight",
            mode="lines",
            line=dict(color="green", width=1.5),
            showlegend=show_legend,
            legendgroup="strategy",
            hovertemplate="<b>%{x|%Y-%m-%d}</b><br>Weight: %{y:.6f}<extra></extra>",
        ),
        row=row, col=col, secondary_y=True,
    )
    fig.add_trace(
        go.Scatter(
            x=xdata, y=dca_weight,
            name=f"Uniform DCA (1/{window_size})",
            mode="lines",
            line=dict(color="#d62728", width=1.5, dash="dash"),
            showlegend=show_legend,
            legendgroup="dca",
            hovertemplate="<b>%{x|%Y-%m-%d}</b><br>DCA Weight: %{y:.6f}<extra></extra>",
        ),
        row=row, col=col, secondary_y=True,
    )


def _style_axes(
    fig,
    row:        int,
    col:        int,
    tick_freq:  Literal["M2", "M12"] = "M2",
    tick_fmt:   Literal["%b", "%Y"]  = "%b",
) -> None:
    """Applies consistent axis styling to a subplot cell."""
    fig.update_yaxes(
        type="log",
        secondary_y=False,
        row=row, col=col,
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
    )
    fig.update_yaxes(
        showgrid=False,
        secondary_y=True,
        row=row, col=col,
    )
    fig.update_xaxes(
        dtick=tick_freq,
        tickformat=tick_fmt,
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        row=row, col=col,
    )


# Data builder

def build_plot_data(
    train_eval_df:  pd.DataFrame,
    train_weights:  np.ndarray,
    test_eval_df:   pd.DataFrame,
    test_weights:   np.ndarray,
    all_win_df: pd.DataFrame,
    train_decay_score: float,
    test_decay_score: float,
    overall_decay_score: float,
    window_size:    int = WINDOW_SIZE,
) -> PlotData:
    """
    Merges train and test into a single DataFrame and computes all
    sats/$ summaries, and attaches the backtest window results. 
    Call once; pass the result to either plot function.
    """
    train_plot = train_eval_df[["date", "price_usd"]].copy()
    train_plot["strategy_weight"] = train_weights
    train_plot["dca_weight"]      = 1.0 / window_size
    train_plot["period"]          = "Train"

    test_plot = test_eval_df[["date", "price_usd"]].copy()
    test_plot["strategy_weight"] = test_weights
    test_plot["dca_weight"]      = 1.0 / window_size
    test_plot["period"]          = "Test"

    df = (
        pd.concat([train_plot, test_plot], ignore_index=True)
        .sort_values("date")
        .reset_index(drop=True)
    )
    df["date"] = pd.to_datetime(df["date"])
    df = df[df["price_usd"] > 0].reset_index(drop=True)
    df["year"] = df["date"].dt.year

    # Train summary
    tr = train_plot[train_plot["price_usd"] > 0]
    train_strat_sats, train_dca_sats, train_excess, train_excess_pct = _sats_summary(
        tr["price_usd"].to_numpy(),
        tr["strategy_weight"].to_numpy(),
        tr["dca_weight"].to_numpy(),
    )

    # Test summary
    te = test_plot[test_plot["price_usd"] > 0]
    test_strat_sats, test_dca_sats, test_excess, test_excess_pct = _sats_summary(
        te["price_usd"].to_numpy(),
        te["strategy_weight"].to_numpy(),
        te["dca_weight"].to_numpy(),
    )

    return PlotData(
        df=df,
        train_eval_df=train_eval_df,
        train_weights=train_weights,
        test_eval_df=test_eval_df,
        test_weights=test_weights,
        all_win_df=all_win_df,
        train_strat_sats=train_strat_sats,
        train_dca_sats=train_dca_sats,
        train_excess=train_excess,
        train_excess_pct=train_excess_pct,
        test_strat_sats=test_strat_sats,
        test_dca_sats=test_dca_sats,
        test_excess=test_excess,
        test_excess_pct=test_excess_pct,
        train_decay_score=train_decay_score,
        test_decay_score=test_decay_score,
        overall_decay_score=overall_decay_score,
    )


# Plot 1: Full period 

def plot_full_period(
    plot_data:   PlotData,
    window_size: int   = WINDOW_SIZE,
    title:       str   = "BTC Price vs Strategy and Uniform DCA Weights",
) -> go.Figure:
    """
    Single-panel plot covering the entire train+test period.
    """
    pd_ = plot_data
    df  = pd_.df

    subtitle = (
        f"Train — Strategy: {pd_.train_strat_sats:,.2f} sats/$ | "
        f"DCA: {pd_.train_dca_sats:,.2f} sats/$ | "
        f"Excess: {_arrow_text_html(pd_.train_excess, pd_.train_excess_pct)}"
        f"<br>"
        f"Test — Strategy: {pd_.test_strat_sats:,.2f} sats/$ | "
        f"DCA: {pd_.test_dca_sats:,.2f} sats/$ | "
        f"Excess: {_arrow_text_html(pd_.test_excess, pd_.test_excess_pct)}"
        f"<br>Exp decay percentile (base 0.9) - Train: {pd_.train_decay_score:.2f} | Test: {pd_.test_decay_score:.2f} | Overall: {pd_.overall_decay_score:.2f}"
    )

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    _add_weight_traces(
        fig,
        xdata=df["date"],
        price=df["price_usd"].to_numpy(),
        strategy_weight=df["strategy_weight"].to_numpy(),
        dca_weight=df["dca_weight"].to_numpy(),
        window_size=window_size,
        row=1, col=1,
        show_legend=True,
    )

    # Test period shading
    fig.add_vrect(
        x0=df.loc[df["period"] == "Test", "date"].min(),
        x1=df["date"].max(),
        fillcolor="lightgreen",
        opacity=0.12,
        line_width=0,
        annotation_text="Test",
        annotation_position="top left",
    )

    _style_axes(fig, row=1, col=1, tick_freq="M12", tick_fmt="%Y")

    fig.update_yaxes(title_text="BTC Price (USD, log scale)", secondary_y=False, row=1, col=1)
    fig.update_yaxes(title_text="Allocation Weight",          secondary_y=True,  row=1, col=1)
    fig.update_xaxes(title_text="Year",                                          row=1, col=1)

    fig.update_layout(
        title=dict(text=f"{title}<br><sup>{subtitle}</sup>", x=0.5),
        template="plotly_white",
        hovermode="x unified",
        legend=dict(orientation="h", y=-0.05),
        margin=dict(l=80, r=90, t=135, b=70),
        height=800,
    )
    return fig


# Plot 2: by backtest window

def plot_by_backtest_window(
    plot_data:   PlotData,
    window_size: int = WINDOW_SIZE,
    n_cols:      int = 2,
    title: str = "BTC Price vs Strategy and Uniform DCA Weights - by Backtest Window",
) -> go.Figure:
    """
    One subplot per backtest window, aligned exactly with all_win_df rows (W1, W2, ...).
    """

    all_win_df = plot_data.all_win_df
    n_windows = len(all_win_df)
    n_rows = math.ceil(n_windows / n_cols)

    # Build subplot titles directly from all_win_df so they match bar chart labels/values exactly.
    subplot_titles = []
    for r in all_win_df.itertuples(index=False):
        start_ym = str(r.start_date)[:7]
        subplot_titles.append(
            f"{r.period} W{r.window}"
            f" {str(r.start_date)[:7]}→{str(r.end_date)[:7]} - SPD ({_arrow_text_html(r.excess_btc * 1e8, r.improvement_pct)})"
        )
    while len(subplot_titles) < n_rows * n_cols:
        subplot_titles.append("")

    specs = [[{"secondary_y": True}] * n_cols for _ in range(n_rows)]
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        specs=specs,
        subplot_titles=subplot_titles,
        vertical_spacing=0.08,
        horizontal_spacing=0.10,
    )

    show_legend = True
    for idx, r in enumerate(plot_data.all_win_df.itertuples(index=False)):
        row = idx // n_cols + 1
        col = idx % n_cols + 1

        if r.period == "Train":
            wdf, ww = _get_window_slice(train_eval_df, train_weights, r.window, window_size)
        else:
            wdf, ww = _get_window_slice(test_eval_df, test_weights, r.window, window_size)

        dca_w = np.full(window_size, 1.0 / window_size)

        _add_weight_traces(
            fig,
            xdata=wdf["date"],
            price=wdf["price_usd"].to_numpy(),
            strategy_weight=ww,
            dca_weight=dca_w,
            window_size=window_size,
            row=row,
            col=col,
            show_legend=show_legend,
        )

        # Shade test windows for visual distinction
        if r.period == "Test":
            fig.add_vrect(
                x0=wdf["date"].min(),
                x1=wdf["date"].max(),
                fillcolor="lightgreen",
                opacity=0.12,
                line_width=0,
                row=row,
                col=col,
            )

        _style_axes(fig, row=row, col=col, tick_freq="M2", tick_fmt="%b")
        show_legend = False

    fig.update_layout(
        title=dict(text=title, x=0.5),
        template="plotly_white",
        hovermode="x unified",
        height=350 * n_rows,
        legend=dict(orientation="h", y=-0.03),
        margin=dict(l=60, r=60, t=80, b=60),
    )
    return fig

# Usage

plot_data = build_plot_data(
    train_eval_df=train_eval_df,
    train_weights=train_weights,
    test_eval_df=test_eval_df,
    test_weights=test_weights,
    all_win_df=all_win_df,
    train_decay_score=train_decay_score,
    test_decay_score=test_decay_score,
    overall_decay_score=overall_decay_score,
    window_size=WINDOW_SIZE,
)

plot_full_period(plot_data).show()
#plot_by_year(plot_data).show()
plot_by_backtest_window(plot_data).show()
